# 03 — SMACS J0723 mini-cutout (real JWST references)

Run this notebook from the torchsky-harness root after installing an explicitly selected torchsky build into the `test` environment. It uses the installed subject and does not add a source checkout to `sys.path`. Install the optional notebook UI once with `pixi run -e test -- python -m pip install jupyterlab matplotlib`, then launch with `pixi run -e test -- jupyter lab notebooks/03_smacs0723_real_minicutout.ipynb`.

This notebook expects multi-band **cropped UPdec mosaic** FITS files produced by:

`pixi run -e test -- python scripts/build_smacs0723_minicutout.py --download`

(Default bands: `F090W`, `F150W`, `F277W`, `F444W` — JWST NIRCam ladder; the repo does not ship HST ACS cutouts, but the workflow is the same once you add FITS inputs.)

Outputs a quick `hashgrid_spectral_field` fit, a rendered hyperspectral cube, and a FITS cube on disk.

In [ ]:
from __future__ import annotations

import os
import tempfile
from pathlib import Path

import torch
import matplotlib.pyplot as plt

import torchfits
from torchsky.field import (
    estimate_domain,
    fit as fit_field_model,
    model as make_model,
    render,
)
from torchsky.scene import FieldObservation
from torchsky.spectral import SpectralAxis, write_cube_fits

# Run Jupyter from the harness root (or set TORCHSKY_DATA).
ROOT = Path.cwd()
data_root = Path(os.environ.get("TORCHSKY_DATA", ROOT / "data")).expanduser().resolve()
cut_dir = data_root / "smacs0723_minicutout"
paths = sorted(cut_dir.glob("smacs0723_*_cutout_*.fits"))
if not paths:
    raise FileNotFoundError(
        f"No cutouts under {cut_dir}. Run:\n"
        "  pixi run -e test -- python scripts/build_smacs0723_minicutout.py --download"
    )

# Approximate effective wavelengths / band support (Å); matches benchmark smoke specs.
BAND_META = {
    "F090W": (9000.0, 8000.0, 10000.0),
    "F150W": (15000.0, 13000.0, 17000.0),
    "F200W": (20000.0, 17000.0, 23000.0),
    "F277W": (27700.0, 24000.0, 32000.0),
    "F356W": (35600.0, 31000.0, 40000.0),
    "F444W": (44400.0, 38000.0, 50000.0),
}

observations = []
ref_header = None
for path in paths:
    band = path.stem.split("_")[1].upper()
    if band not in BAND_META:
        continue
    wave, smin, smax = BAND_META[band]
    data, hdr = torchfits.read(str(path), hdu=0, return_header=True)
    if data.ndim != 2:
        data = data.reshape(data.shape[-2], data.shape[-1])
    hdr_d = dict(hdr)
    ref_header = hdr_d
    observations.append(
        FieldObservation(
            image=data.to(dtype=torch.float32),
            header=hdr_d,
            wavelength=float(wave),
            spectral_min=float(smin),
            spectral_max=float(smax),
            transmission_center=float(wave),
            band_name=band,
        )
    )

if len(observations) < 2:
    raise RuntimeError("Need at least two band cutouts; check filenames and BAND_META.")

domain = estimate_domain(observations)
torch.manual_seed(13)
model = make_model(
    "hashgrid_spectral_field",
    domain,
    hidden_dim=48,
    num_levels=4,
    log2_hashmap_size=14,
)
fit = fit_field_model(
    observations,
    model,
    steps=80,
    lr=2e-3,
    loss_kind="charbonnier",
    num_spectral_samples=10,
)
print("bands", [o.band_name for o in observations])
print("loss", fit.initial_loss, "->", fit.final_loss)

bands_loaded = [o.band_name for o in observations if o.band_name]
lo = min(BAND_META[b][1] for b in bands_loaded)
hi = max(BAND_META[b][2] for b in bands_loaded)
# Reasonable λ grid across loaded bands
waves = torch.linspace(float(lo), float(hi), 24, dtype=torch.float64)
cube_axis = SpectralAxis(values=waves, unit="Angstrom", type="WAVE")

with torch.no_grad():
    cube = render(model, target="cube", target_header=ref_header, spectral_axis=cube_axis)

tmpdir = Path(tempfile.mkdtemp(prefix="torchsky_nb03_"))
out_path = tmpdir / "smacs_minicut_hyperspectral.fits"
spatial_hdr = {
    k: v
    for k, v in ref_header.items()
    if not str(k)
    .upper()
    .startswith(("NAXIS", "CTYPE3", "CRVAL3", "CDELT3", "CRPIX3", "CUNIT3"))
}
spatial_hdr["NAXIS"] = 2
spatial_hdr["NAXIS1"] = ref_header["NAXIS1"]
spatial_hdr["NAXIS2"] = ref_header["NAXIS2"]
write_cube_fits(
    str(out_path),
    cube.cpu(),
    spatial_header=spatial_hdr,
    spectral_axis=cube_axis,
    overwrite=True,
)
print("wrote", out_path)

# Pseudo-RGB from three wavelength slices (normalize per channel for display).
idx = [
    max(0, len(waves) // 6),
    len(waves) // 2,
    min(len(waves) - 1, (5 * len(waves)) // 6),
]


def norm(x: torch.Tensor) -> torch.Tensor:
    x = x - x.min()
    return x / (x.max() + 1e-8)


r, g, b = tuple(norm(cube[i]) for i in idx)
rgb = torch.stack([r, g, b], dim=0).permute(1, 2, 0).cpu().numpy()
plt.figure(figsize=(4, 4))
plt.imshow(rgb, origin="lower")
plt.title("pseudo-RGB from cube slices")
plt.axis("off")
plt.tight_layout()
plt.show()

cy, cx = cube.shape[1] // 2, cube.shape[2] // 2
plt.figure(figsize=(6, 3))
plt.plot(waves.numpy(), cube[:, cy, cx].cpu().numpy())
plt.xlabel("wavelength (Angstrom)")
plt.ylabel("model intensity (arb.)")
plt.title(f"spectrum at pixel ({cy}, {cx})")
plt.tight_layout()
plt.show()